## tl;dr
Four previously uncurated projects now have 8 documented interfaces, 11 resources and 16 scoped curated facts. The same 30 deterministic tasks passed 3 → 23 → 30 across the saved before / data-only / after runs. These are evaluator-curated contract tasks, not independent agent trials or a market-demand result.

## Context & Methods
This notebook is a companion audit of the local enrichment batch on August 28, 2026. Original HTTP requests and checks are produced by `evaluations/knowledge-expansion.ts` against isolated SQLite; all response bodies are preserved in the three receipts. No third-party executable, robot or production database is used.

### Key Assumptions
The cohort is OpenHands, LangGraph, LeRobot and Playwright MCP. It is no longer a holdout once curated. Task IDs, expectations and evaluation clock are identical across phases. Before vs data-only changes only intake data; data-only vs after changes the query code. GitHub observations pin source text but do not establish runtime compatibility, fresh-field TTL or full historical reconstruction. Initial fact additions are observations/intake audits, not complete upstream change events.

## Data
### 1. Load preserved receipts and the frozen task specification
Run from the repository root or this notebook's directory. Python's standard library is sufficient for the cells. A Jupyter runtime is only required to open/execute through the notebook UI.

In [1]:
from pathlib import Path
import json, hashlib, sqlite3

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'evaluations/knowledge-expansion-tasks.json').is_file())
folder = root / 'docs/evaluations/2026-08-28-expansion'
stages = ['before', 'data-only', 'after']
texts = [(folder / f'{stage}.json').read_text() for stage in stages]
receipts = [json.loads(text) for text in texts]
spec_path = root / 'evaluations/knowledge-expansion-tasks.json'
spec = json.loads(spec_path.read_text())
print({'stages': stages, 'tasks': len(spec['tasks']), 'cohort': spec['cohort'], 'evaluationTime': spec['evaluationTime']})


{'stages': ['before', 'data-only', 'after'], 'tasks': 30, 'cohort': ['openhands', 'langgraph', 'lerobot', 'playwright-mcp'], 'evaluationTime': '2026-08-28T05:44:00.000Z'}


### 2. Reconcile identities, fingerprints and pass/fail results
Validate raw tasks rather than trusting the saved summary counters. Require unchanged task/evaluator fingerprints, unchanged data between the last two phases, and no changed project identity.

In [2]:
digest = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
expected_ids = [task['id'] for task in spec['tasks']]
assert len(expected_ids) == len(set(expected_ids)) == 30
for receipt in receipts:
    assert receipt['evaluationTime'] == spec['evaluationTime']
    assert receipt['fingerprints']['taskSpec'] == digest(spec_path)
    assert [task['id'] for task in receipt['tasks']] == expected_ids
    passed = sum(task['passed'] for task in receipt['tasks'])
    assert passed == receipt['summary']['passed']
    assert [task['id'] for task in receipt['tasks'] if not task['passed']] == receipt['summary']['failed']
assert len({r['fingerprints']['evaluator'] for r in receipts}) == 1
assert receipts[0]['fingerprints']['query'] == receipts[1]['fingerprints']['query']
assert receipts[1]['fingerprints']['query'] != receipts[2]['fingerprints']['query']
assert receipts[1]['fingerprints']['manifests'] == receipts[2]['fingerprints']['manifests']
assert digest(root / 'evaluations/knowledge-expansion.ts') == receipts[2]['fingerprints']['evaluator']
assert digest(root / 'src/lib/registry/knowledge-query.ts') == receipts[2]['fingerprints']['query']
for slug, expected in receipts[2]['fingerprints']['manifests'].items():
    assert digest(root / f'content/intake/{slug}.json') == expected
assert [[p['projectId'] for p in r['coverage']] for r in receipts].count([p['projectId'] for p in receipts[0]['coverage']]) == 3
print({'passCounts': [r['summary']['passed'] for r in receipts], 'taskAndDataFingerprints': 'matched', 'identities': 'unchanged'})


{'passCounts': [3, 23, 30], 'taskAndDataFingerprints': 'matched', 'identities': 'unchanged'}


## Results
### 3. Aggregate the same tasks by question group
The SQL takes the three full receipt JSON strings as positional parameters. Each group retains its denominator; rates are fractions, not percentage-point inputs.

In [3]:
connection = sqlite3.connect(':memory:')
connection.row_factory = sqlite3.Row
query = (folder / 'task-outcomes.sql').read_text()
rows = [dict(row) for row in connection.execute(query, texts)]
connection.close()
assert len(rows) == 18
for stage in ['补数前', '仅补数', '迭代后']:
    selected = [row for row in rows if row['stage'] == stage]
    assert sum(row['total'] for row in selected) == 30
    print(stage, {row['taskGroup']: f"{row['passed']}/{row['total']}" for row in selected})


补数前 {'事实证据': '0/9', '历史边界': '1/1', '字段发现': '0/4', '接口查询': '0/7', '版本资源': '0/2', '边界判断': '2/7'}
仅补数 {'事实证据': '9/9', '历史边界': '1/1', '字段发现': '0/4', '接口查询': '4/7', '版本资源': '2/2', '边界判断': '7/7'}
迭代后 {'事实证据': '9/9', '历史边界': '1/1', '字段发现': '4/4', '接口查询': '7/7', '版本资源': '2/2', '边界判断': '7/7'}


### 4. Count the complete field-discovery sequence
Include every paginated index request and the final fact request. These are uncompressed JSON bytes, not tokens, network latency or lossless compression.

In [4]:
after = receipts[2]
sequences = []
for task in after['tasks']:
    if task['id'].startswith('X'):
        sequences.append({'task': task['id'], 'requests': len(task['responses']), 'jsonBytes': sum(r['jsonBytes'] for r in task['responses'])})
print(sequences)
print({key: sum(p[key] for p in after['coverage']) for key in ['interfaces', 'resources', 'curatedFacts', 'runtimeTests', 'scopedFreshnessChecks', 'projectionIssues']})
assert all(p['runtimeTests'] == p['scopedFreshnessChecks'] == p['projectionIssues'] == 0 for p in after['coverage'])


[{'task': 'X1', 'requests': 4, 'jsonBytes': 7483}, {'task': 'X2', 'requests': 4, 'jsonBytes': 7320}, {'task': 'X3', 'requests': 5, 'jsonBytes': 9890}, {'task': 'X4', 'requests': 4, 'jsonBytes': 7394}]
{'interfaces': 8, 'resources': 11, 'curatedFacts': 16, 'runtimeTests': 0, 'scopedFreshnessChecks': 0, 'projectionIssues': 0}


## Takeaways
- Data enrichment resolves 20 previously failing tasks; domain filtering and field-key discovery resolve the remaining 7. Three boundary-only tasks already passed without enrichment.
- Discover-then-fetch takes 4–5 requests and 7,320–9,890 JSON bytes in these four cases. This includes discovery overhead but does not establish real-agent cost savings.
- Continue a bounded product validation. Independent clients, new unseen projects, actual retention/backfill, freshness review and safe scoped runtime reports remain unvalidated. Do not treat 30/30 as general accuracy.

Validation mode: the code cells are run in order with a shared Python namespace and captured stdout. Notebook structure is checked using the standard library. Jupyter kernel/UI execution is not verified because Jupyter, nbformat and nbclient are absent. Once a Jupyter environment is available, run `jupyter nbconvert --execute --to notebook --inplace docs/evaluations/2026-08-28-expansion/analysis.ipynb`.